# LangGraph MCP Bash 튜토리얼

이 튜토리얼에서는 LangGraph와 MCP(Model Context Protocol)를 활용하여 **Bash 명령어 실행**이 가능한 AI 에이전트를 구축합니다. `mcp-bash` 서버는 LLM이 Bash 명령어를 직접 실행하고 결과를 받아볼 수 있도록 해주는 MCP 서버입니다.

> 참고 자료: [mcp-bash GitHub](https://github.com/patrickomatik/mcp-bash)

### mcp-bash 주요 도구

| 도구 | 설명 |
|------|------|
| `set_cwd` | 이후 명령어 실행에 사용할 작업 디렉토리(CWD)를 설정합니다 |
| `execute_bash` | Bash 명령어를 실행하고 stdout/stderr 결과를 반환합니다 |

### 사전 준비

mcp-bash 서버를 실행하려면 먼저 저장소를 클론하고 설치해야 합니다.

```bash
# 1. 저장소 클론
git clone https://github.com/patrickomatik/mcp-bash.git
cd mcp-bash

# 2. 가상 환경 생성 및 설치
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
pip install -e .
```

### ⚠️ 보안 경고

> **주의**: mcp-bash는 Bash 명령어를 직접 실행합니다. 서버 제작자 스스로 *"치명적인 보안 위험(lethal security risk)"*이 될 수 있다고 경고하고 있습니다. 이 서버는 명령어 실행에 대한 제한이나 안전장치가 없습니다.
>
> **안전한 사용을 위한 권장 사항:**
> - 신뢰할 수 있는 **로컬 개발 환경**에서만 사용하세요.
> - **컨테이너**(Docker 등) 내에서 실행하여 호스트 시스템을 보호하세요.
> - 프로덕션 환경이나 외부에 노출된 시스템에서는 절대 사용하지 마세요.
> - 허용할 명령어 목록(allowlist)을 별도로 구현하는 것을 권장합니다.

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 설정을 활성화하여 LangSmith에서 실행 결과를 확인할 수 있도록 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 프로젝트 추적 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

## 라이브러리 임포트 및 MCP 클라이언트 설정

LangGraph 에이전트 구성에 필요한 라이브러리를 임포트하고, MCP 클라이언트 설정을 위한 유틸리티 함수를 정의합니다.

In [ ]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 여러 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호환성 활성화 (Jupyter 환경에서 필요)
nest_asyncio.apply()

In [ ]:
import sys, os, subprocess

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리. 각 서버의 이름을 키로,
                       연결 정보(command, args, transport 또는 url)를 값으로 가집니다.

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    # MCP 클라이언트 생성
    client = MultiServerMCPClient(server_configs)

    # 서버에 연결하여 도구 목록을 가져옵니다
    tools = await client.get_tools()

    # 로드된 도구 목록을 출력합니다
    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")

    return client, tools

---

## Part 1: mcp-bash 서버 연결

mcp-bash 서버를 stdio 전송 방식으로 연결합니다. `uv`를 사용하여 `mcp[cli]` 패키지와 함께 `server.py`를 실행합니다.

아래 코드에서 `MCP_BASH_PATH` 변수를 클론한 mcp-bash 저장소의 `server.py` 경로로 변경하세요.

In [ ]:
import os

# mcp-bash 저장소의 server.py 경로를 지정합니다 (본인의 환경에 맞게 수정하세요)
MCP_BASH_PATH = os.path.abspath("./mcp-bash/server.py")

print(f"mcp-bash server.py 경로: {MCP_BASH_PATH}")
print(f"파일 존재 여부: {os.path.exists(MCP_BASH_PATH)}")

In [ ]:
# mcp-bash 서버 설정 (stdio 전송 방식)
server_configs = {
    "bash": {
        "command": "uv",
        "args": ["run", "--with", "mcp[cli]", "mcp", "run", MCP_BASH_PATH],
        "transport": "stdio",
    }
}

# MCP 클라이언트 생성 및 도구 로드
client, tools = await setup_mcp_client(server_configs)

---

## Part 2: Bash 에이전트 생성

mcp-bash 도구를 사용하는 에이전트를 생성합니다. `create_agent`는 LangChain에서 제공하는 에이전트 생성 함수로, LLM과 도구를 결합하여 추론-행동 루프를 자동으로 구성합니다.

> 참고: LangGraph v1에서 기존의 `create_react_agent`는 deprecated 되었으며, `langchain.agents.create_agent`를 사용하는 것이 권장됩니다.

In [ ]:
# LLM 설정
# OpenAI 키 사용 시 gpt-4.1, gpt-4.1-mini 등으로 변경 가능
llm = init_chat_model("claude-sonnet-4-5", temperature=0)

# 에이전트 생성: mcp-bash 도구를 사용하는 에이전트
agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
)

In [ ]:
# 스트리밍 출력 함수와 UUID 생성 함수를 import합니다
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

---

## Part 3: 현재 디렉토리 확인

`execute_bash` 도구를 통해 `pwd` 명령어를 실행하여 서버의 현재 작업 디렉토리를 확인합니다. 이를 통해 mcp-bash 서버가 정상적으로 연결되었는지 검증할 수 있습니다.

In [ ]:
# 대화 스레드 ID를 생성합니다
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 현재 작업 디렉토리 확인
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "현재 작업 디렉토리가 어디인지 알려주세요.")]},
    config=config,
)

---

## Part 4: 작업 디렉토리 설정 및 파일 목록 조회

`set_cwd` 도구로 작업 디렉토리를 변경한 후 `execute_bash`로 `ls` 명령어를 실행합니다. `set_cwd`로 설정한 디렉토리는 이후 모든 `execute_bash` 호출에 자동으로 적용됩니다.

In [ ]:
import os

# 작업 디렉토리로 사용할 경로 (본인의 환경에 맞게 수정하세요)
WORK_DIR = os.path.abspath("./workspace")
os.makedirs(WORK_DIR, exist_ok=True)

print(f"작업 디렉토리: {WORK_DIR}")

In [ ]:
# 새로운 대화 스레드 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 작업 디렉토리 설정 후 파일 목록 조회
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                f"작업 디렉토리를 '{WORK_DIR}'로 설정한 다음, "
                "해당 디렉토리의 파일 목록을 자세히 보여주세요. (ls -la 명령어 사용)",
            )
        ]
    },
    config=config,
)

---

## Part 5: 파일 생성 및 내용 작성

에이전트에게 Bash 명령어를 이용한 파일 생성을 요청합니다. `echo`, `cat`, `printf` 등의 명령어를 활용하여 파일을 만들고 내용을 작성할 수 있습니다.

In [ ]:
# 에이전트 실행: Python 스크립트 파일 생성 (이전 스레드에서 CWD 유지)
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "현재 작업 디렉토리에 'greet.py' 라는 Python 파일을 만들어주세요. "
                "파일 내용은 다음과 같습니다:\n"
                "def greet(name):\n"
                "    return f'안녕하세요, {name}님! LangGraph MCP Bash 튜토리얼에 오신 것을 환영합니다.'\n\n"
                "if __name__ == '__main__':\n"
                "    print(greet('사용자'))",
            )
        ]
    },
    config=config,
)

---

## Part 6: 스크립트 실행 및 결과 확인

생성한 Python 스크립트를 `execute_bash`로 실행하고 결과를 확인합니다. 이는 mcp-bash의 핵심 사용 사례로, **코드 작성 → 실행 → 결과 확인**을 하나의 에이전트 대화에서 처리할 수 있습니다.

In [ ]:
# 에이전트 실행: 생성한 스크립트 실행 (같은 스레드에서 CWD 유지)
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "방금 만든 'greet.py' 파일을 Python으로 실행하고 결과를 알려주세요. "
                "실행 전에 파일 내용도 cat으로 확인해주세요.",
            )
        ]
    },
    config=config,
)

---

## Part 7: 개발 워크플로우 (코드 작성 → 실행 → 수정)

mcp-bash의 가장 강력한 활용 사례는 **자율적인 개발 워크플로우**입니다. 에이전트가 코드를 작성하고, 실행하여 오류를 확인하고, 자동으로 수정하는 전체 사이클을 하나의 대화에서 처리할 수 있습니다.

아래는 간단한 수학 유틸리티 함수를 작성하고 테스트하는 예시입니다.

In [ ]:
# 새로운 대화 스레드 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 코드 작성 → 테스트 실행 → 오류 수정 워크플로우
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                f"작업 디렉토리를 '{WORK_DIR}'로 설정하고, 다음 단계를 순서대로 수행해주세요:\n\n"
                "1. 'math_utils.py' 파일을 생성하세요. 내용은 다음과 같습니다:\n"
                "   - 소수(prime number) 여부를 판별하는 is_prime(n) 함수\n"
                "   - 피보나치 수열의 n번째 값을 반환하는 fibonacci(n) 함수\n\n"
                "2. 'test_math_utils.py' 테스트 파일을 생성하세요. 다음을 테스트해야 합니다:\n"
                "   - is_prime: 2, 7, 13은 소수 / 1, 4, 9는 소수 아님\n"
                "   - fibonacci: f(0)=0, f(1)=1, f(10)=55\n\n"
                "3. 'python -m pytest test_math_utils.py -v' 명령어로 테스트를 실행하세요.\n\n"
                "4. 테스트가 실패하면 코드를 수정하고 다시 실행하세요. 모든 테스트가 통과할 때까지 반복하세요.",
            )
        ]
    },
    config=config,
)

---

## Part 8: 환경 정보 수집

`execute_bash`를 활용하면 시스템 환경 정보를 손쉽게 수집할 수 있습니다. 운영체제 정보, Python 버전, 설치된 패키지, 디스크 사용량 등 다양한 시스템 상태를 조회하는 데 활용할 수 있습니다.

In [ ]:
# 새로운 대화 스레드 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 시스템 환경 정보 수집
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "현재 시스템 환경 정보를 수집해주세요. 다음 항목을 포함해주세요:\n"
                "1. 운영체제 정보 (uname -a 또는 해당 OS에 맞는 명령어)\n"
                "2. Python 버전 (python --version 또는 python3 --version)\n"
                "3. pip으로 설치된 주요 패키지 목록 (pip list | head -20)\n"
                "4. 현재 디렉토리의 디스크 사용량 (du -sh . 또는 dir)\n"
                "수집한 정보를 정리하여 요약해주세요.",
            )
        ]
    },
    config=config,
)

---

## 보안 주의사항 요약

mcp-bash는 강력한 도구이지만 그만큼 주의가 필요합니다. 다음 사항을 반드시 숙지하세요:

| 위험 | 설명 |
|------|------|
| **임의 명령어 실행** | 에이전트가 `rm -rf /`, `chmod 777`, `curl \| bash` 등 위험한 명령어를 실행할 수 있습니다 |
| **안전장치 없음** | mcp-bash 서버 자체에는 명령어를 제한하는 메커니즘이 없습니다 |
| **LLM 판단 의존** | 명령어 실행 여부는 전적으로 LLM의 판단에 달려 있습니다 |

**권장 보안 조치:**

1. **컨테이너화**: Docker 등을 사용하여 격리된 환경에서 실행
2. **명령어 허용 목록(Allowlist)**: `execute_bash` 도구에 명령어 필터링 로직 추가
3. **파일 시스템 권한 제한**: 에이전트가 접근할 수 있는 디렉토리와 권한을 최소화
4. **신뢰할 수 있는 환경 전용**: 로컬 개발 환경에서만 사용하고 외부에 절대 노출하지 않음
5. **Human-in-the-loop**: 중요한 작업 전 사용자 승인 단계 추가 고려

---

## 정리

이 튜토리얼에서 다룬 내용을 정리합니다:

1. **mcp-bash 서버 연결**: `uv run --with mcp[cli] mcp run server.py`로 실행하고 `MultiServerMCPClient`로 연결
2. **기본 명령어 실행**: `execute_bash`를 통해 `pwd`, `ls` 등 기본 명령어 실행
3. **작업 디렉토리 설정**: `set_cwd`로 CWD를 변경하면 이후 모든 명령어에 자동 적용
4. **파일 생성 및 실행**: Bash 명령어로 파일을 생성하고 Python 스크립트를 실행
5. **자율 개발 워크플로우**: 코드 작성 → 테스트 실행 → 오류 수정의 전체 사이클 자동화
6. **시스템 정보 수집**: 다양한 Bash 명령어로 환경 정보를 수집하고 정리

mcp-bash를 활용하면 에이전트가 **코드를 작성하고, 즉시 실행하여 결과를 확인하고, 문제를 자동으로 해결**하는 강력한 개발 자동화 파이프라인을 구축할 수 있습니다.